# End-to-end workflow with Enterprise h2oGPTe

Get started building generative AI applications with Enterprise h2oGPTe in the H2O AI Cloud.

* Connect to Enterprise h2oGPTe
* Create a collection and ingest documents for retrieval augmented generation (RAG)
* Chat with your documents and tune the RAG options
* Analyze audio, video and images
* Use agents to complete multi-step tasks
* Connect your own [Model Context Protocol](https://modelcontextprotocol.io) (MCP) server to give agents new tools
* Schedule recurring tasks

Python Client Docs : https://h2oai.github.io/h2ogpte/index.html

## Connect to Enterprise h2oGPTe
Enterprise h2oGPTe is a managed service, so there is no AI engine to create - connect with the client and start working.

In [1]:
from h2o_ai_cloud import h2ogpte_client

h2ogpte = h2ogpte_client()

Visit https://internal.dedicated.h2o.ai/auth/get-platform-token to get your platform token


### List the available LLMs

Enterprise h2oGPTe is LLM-agnostic and provides access to a range of large language models.

Choose one to use for the rest of the tutorial or alternatively, let Enterprise h2oGPTe selecting automatically the LLM best for each task.

In [2]:
llm_names = h2ogpte.get_llm_names()
llm_names

['amazon/nova-2-lite',
 'claude-haiku-4-5',
 'claude-opus-4-6',
 'claude-opus-4-7',
 'claude-opus-4-8',
 'claude-opus-5',
 'claude-opus-5-5',
 'claude-sonnet-4-6',
 'claude-sonnet-5',
 'claude-sonnet-5-5',
 'deepseek-ai/DeepSeek-R1',
 'deepseek-ai/DeepSeek-V3.1',
 'deepseek-ai/DeepSeek-V3.2',
 'google/gemma-3-12b-it',
 'google/gemma-3-27b-it',
 'google/gemma-3-4b-it',
 'gpt-5.6-luna',
 'gpt-5.6-sol',
 'gpt-5.6-terra',
 'gpt-6-astra',
 'meta-llama/Llama-3.3-70B-Instruct',
 'meta-llama/Llama-4-Maverick-17B-128E-Instruct',
 'meta-llama/Llama-4-Scout-17B-16E-Instruct',
 'meta-llama/Meta-Llama-3.1-70B-Instruct',
 'meta-llama/Meta-Llama-3.1-8B-Instruct',
 'MiniMaxAI/MiniMax-M2',
 'MiniMaxAI/MiniMax-M2.1',
 'MiniMaxAI/MiniMax-M2.5',
 'mistralai/Devstral-2-123B',
 'mistralai/Magistral-Small-2509',
 'mistralai/Ministral-3-14B-Instruct',
 'mistralai/Ministral-3-3B-Instruct',
 'mistralai/Ministral-3-8B-Instruct',
 'mistralai/Mistral-7B-Instruct-v0.2',
 'mistralai/Mistral-Large-3-675B-Instruct',
 

"auto" lets Enterprise h2oGPTe pick the best available LLM, or choose a name from the list above:

In [ ]:
llm = "auto"

## Build a RAG (Retrieval Augmented Generation) application

### Create a collection
A collection is a group of documents that h2oGPTe indexes so the LLM can retrieve relevant content when answering questions.

In [4]:
collection_id = h2ogpte.create_collection(
    name="H2O AI Cloud Tutorial",
    description="Documentation used to answer questions about Enterprise h2oGPTe"
)

print("Collection ID:", collection_id)

Collection ID: c22b6716-dd23-4db7-b2c1-6af1ca06123b


### Import data
Ingest documents into the collection. This tutorial ingests a public web page, but you can also ingest your own files with `h2ogpte.upload` and `h2ogpte.ingest_uploads`.

In [6]:
h2ogpte.ingest_website(
    collection_id=collection_id,
    url="https://docs.h2o.ai/enterprise-h2ogpte/",
    follow_links=False,
    gen_doc_summaries=True
)

Job(id='a4f81e76-f7ff-4ed3-ac06-2b7ab2f956b8', name='Ingesting from web', passed=1.0, failed=0.0, progress=1.0, completed=True, canceled=False, date=datetime.datetime(2026, 10, 12, 14, 30, 48, tzinfo=TzInfo(0)), kind=<JobKind.IngestWebsiteJob: 'IngestWebsiteJob'>, statuses=[JobStatus(id='93041e82502a4535b426521b1f11de75', status='Cleaning up...'), JobStatus(id='d4ced6bfc69d45a5be288e69bafd96dd', status='Indexing done.')], errors=[], last_update_date=datetime.datetime(2026, 10, 12, 14, 35, 59, tzinfo=TzInfo(0)), duration='5m11s', duration_seconds=311.0, start_time=1791815725.7117903, canceled_by='', cancel_reason='', timeout=10800.0, files_total=None, files_failed=None)

In [7]:
documents = h2ogpte.list_documents_in_collection(collection_id, offset=0, limit=100)

for doc in documents:
    print(doc.id, "|", doc.name)

27424dc1-47c4-4750-b82a-275e0a28c5b4 | Enterprise h2oGPTe Docs Enterprise h2oGPTe.pdf
d8fca9df-ca29-4847-95d9-b68736569f40 | Enterprise h2oGPTe Docs Enterprise h2oGPTe.pdf


## Chat with your documents

### Create a chat session
A chat session keeps the conversation history, and is linked to the collection so that answers are grounded in your documents.

In [8]:
chat_session_id = h2ogpte.create_chat_session(collection_id)

print("Chat session ID:", chat_session_id)

Chat session ID: 10a528f8-10b4-44d4-80a5-86035d58b98c


### Ask a question

In [9]:
with h2ogpte.connect(chat_session_id) as session:
    answer = session.query(
        "What is Enterprise h2oGPTe and what can I use it for?",
        llm=llm,
        rag_config={"rag_type": "rag"},  # retrieve relevant chunks from the collection
    )

print(answer.content)

According to the provided documentation, I can share the following about Enterprise h2oGPTe, though it's important to note that the documents consist primarily of navigation links and page structure rather than detailed descriptive content.

**What is Enterprise h2oGPTe?**

Enterprise h2oGPTe is a product created by H2O.ai, with the current version being **v1.7.6-6** (with v1.7.4 also referenced as a recent release). The documentation positions it as a tool designed to help users **"Find information 10x faster."** For a complete definition, the documentation directs users to a dedicated page titled https://docs.h2o.ai/enterprise-h2ogpte/get-started/what-is-h2ogpte, though the specific content of that page is not included in the provided text.

**What can you use it for?**

Based on the features and capabilities referenced throughout the documentation structure, Enterprise h2oGPTe supports the following uses:

- **Chats** – Conduct chat sessions, customize them, share them, and chat dir

### Ask a follow-up question
The chat session remembers the conversation, so follow-up questions can build on previous answers. Retrieval settings such as the number of chunks to use are configured with `rag_config`.

In [10]:
with h2ogpte.connect(chat_session_id) as session:
    answer = session.query(
        "Summarize that in three bullet points.",
        llm=llm,
        rag_config={"rag_type": "rag"}
    )

print(answer.content)

According to the provided documentation:

- **What it is:** Enterprise h2oGPTe is an H2O.ai product (current version v1.7.6-6) designed to help users **"Find information 10x faster."**

- **Core capabilities:** It supports Chats, Collections, Documents, Models, Prompts, Extractors, Evaluation, Feedback, Workspaces, and Roles/Permissions — enabling users to chat with document collections, generate document summaries, run evaluators, extract information, and manage access.

- **Integration & resources:** It offers a REST API (Swagger UI), SDKs, and a Python Client Library for programmatic use, along with supporting resources such as tutorials, FAQs, training/learning paths, and architecture documentation.


### More RAG options
`rag_config` controls how h2oGPTe uses your documents. The right `rag_type` available

| `rag_type` | What it does | Use it for |
|---|---|---|
| `auto` | h2oGPTe chooses | Default when unsure |
| `llm_only` | No retrieval | General knowledge, baseline for comparison |
| `rag` | Retrieve the top chunks, answer from them | Fast factual lookups |
| `hyde1` / `hyde2` | Draft an answer first, then use it to improve retrieval | Vague or short questions |
| `rag+` | Retrieve, add neighbouring chunks, summarize recursively | Questions needing more context |
| `all_data` | Read every chunk of the collection | Whole-document summaries |
| `agentic_rag` | An agent calls the RAG tool as many times as it needs | Multi-step questions |
| `rlm_rag` | An agent analyses documents with Python and sub-LLM calls | Multi-hop reasoning over large documents |


### Compare RAG types on the same question

In [12]:
question = "Which features help me evaluate the quality of answers?"

for rag_type in ["llm_only", "rag", "hyde1", "rag+", "agentic_rag"]:
    with h2ogpte.connect(chat_session_id) as session:
        reply = session.query(
            question,
            llm=llm,
            rag_config={"rag_type": rag_type},
            include_chat_history="off",  # keep each run independent
            timeout=600,
        )
    print(f"=== {rag_type} ===")
    print(reply.content[:500], "\n")

=== llm_only ===
Based on the collection "H2O AI Cloud Tutorial" available to me, I can provide general guidance on features that help evaluate answer quality in Enterprise h2oGPTe. However, I notice that no specific document excerpts have been shared with me in this query — only the collection metadata.

That said, here are features commonly available in Enterprise h2oGPTe to help evaluate answer quality:

1. **Self-Reflection / Answer Scoring** - h2oGPTe can rate its own answers for quality, giving you a confi 

=== rag ===
According to the information provided in the documents, Enterprise h2oGPTe offers several features that help you evaluate the quality of answers:

## Evaluation Features

### 1. **Evaluators**
- **Run an Evaluator on a Collection** — Allows you to evaluate the quality of answers across an entire Collection of documents.
- **Run an Evaluator on a Chat** — Enables you to assess the quality of answers within a specific Chat session.

### 2. **Evaluation Dashboards**


### Tune retrieval
Control how many chunks are retrieved, the minimum relevance score, and how much surrounding context is added to each chunk.

In [20]:
with h2ogpte.connect(chat_session_id) as session:
    tuned = session.query(
        "List the ways to integrate Enterprise h2oGPTe programmatically.",
        llm=llm,
        rag_config={
            "rag_type": "rag+",
            "rag_max_chunks": 20,                 # retrieve at most 20 chunks
            "rag_min_chunk_score": 0.2,           # drop weakly related chunks
            "num_neighbor_chunks_to_include": 2,  # add 2 neighbours around each hit (rag+ only)
            "meta_data_to_include": {"name": True, "page": True, "text": True, "uri": True},
        },
        include_chat_history="off",
        timeout=600,
    )

print(tuned.content)

According to the information provided in the documents, Enterprise h2oGPTe can be integrated programmatically through the following ways:

## API & SDKs

1. **h2oGPTe REST API: Interactive API docs (Swagger UI)**
   - Available at: https://h2ogpte.genai.h2o.ai/swagger-ui/
   - Provides interactive API documentation for exploring and testing the REST API endpoints.

2. **h2oGPTe REST API: SDKs**
   - Documentation available at: https://docs.h2o.ai/enterprise-h2ogpte/rest-api/sdks
   - Software Development Kits that allow developers to interact with the h2oGPTe REST API.

3. **h2oGPTe Python Client Library**
   - Available at: https://h2oai.github.io/h2ogpte/index.html
   - A dedicated Python client library for programmatic integration with h2oGPTe.

## API Keys Management

To support programmatic access, Enterprise h2oGPTe also provides API key management capabilities:

- **Create an API key** — https://docs.h2o.ai/enterprise-h2ogpte/guide/apis#create-an-api-key
- **Delete an API Key** 

### Get structured output
Ask for JSON to feed the answer straight into code. Use `guided_json` with a JSON schema on models that support guided generation.

In [21]:
import json

with h2ogpte.connect(chat_session_id) as session:
    structured = session.query(
        "List 3 capabilities of Enterprise h2oGPTe as a JSON object with a 'capabilities' list, "
        "each item having 'name' and 'description'.",
        llm=llm,
        llm_args={"response_format": "json_object"},
        rag_config={"rag_type": "rag"},
        include_chat_history="off",
    )

print(json.dumps(json.loads(structured.content), indent=2))

{
  "response": "According to the information provided in the documents, here are 3 capabilities of Enterprise h2oGPTe:",
  "capabilities": [
    {
      "name": "Chat with a Collection",
      "description": "Enterprise h2oGPTe allows users to interact with a Collection of documents through a chat interface, enabling conversational queries against document content."
    },
    {
      "name": "Document Summary Generation",
      "description": "The platform can generate, view, and delete summaries of documents, helping users quickly understand document contents without reading them in full."
    },
    {
      "name": "Extractors for Documents",
      "description": "Users can create and run Extractors on documents to pull specific information, enabling structured data extraction from unstructured document content."
    }
  ]
}


### Let h2oGPTe control cost
With `llm="auto"`, `cost_controls` lets the router pick the cheapest model that is accurate enough.

In [22]:
with h2ogpte.connect(chat_session_id) as session:
    routed = session.query(
        "In one sentence, what is a collection?",
        llm="auto",
        llm_args={
            "cost_controls": {
                "max_cost_per_million_tokens": 10,  # only consider models under $10 per million tokens
                "willingness_to_pay": 0.05,         # USD per +10% accuracy
            }
        },
        rag_config={"rag_type": "rag"},
        include_chat_history="off",
    )

print(routed.content)

<think> The user is asking: "In one sentence, what is a collection?" based only on the information provided in the documents.

I need to find any mention of "collection" in the documents and define it in one sentence.

Looking at the documents:

From page 2 of both PDFs:
- "apps Collections" appears as a navigation item
- "Create a Collection (Link: https://docs.h2o.ai/enterprise-h2ogpte/guide/collections/create-a-collection)"
- "Add a Document(s) to a Collection (Link: https://docs.h2o.ai/enterprise-h2ogpte/guide/collections/add-documents-to-a-collection)"
- "Chat with a Collection (Link: https://docs.h2o.ai/enterprise-h2ogpte/guide/collections/chat-with-a-collection)"
- "Run an Evaluator on a Collection (Link: https://docs.h2o.ai/enterprise-h2ogpte/guide/collections/evaluate-a-collection)"
- "Supported file types for a Collection (Link: https://docs.h2o.ai/enterprise-h2ogpte/guide/collections/supported-file-types)"
- "Update a Collection's settings (Link: https://docs.h2o.ai/enterpri

## Analyze audio, video and images
Enterprise h2oGPTe ingests more than text. Audio, and the audio track of video, is transcribed to text so it can be searched, summarized and questioned like any other document. Images and scanned pages go through OCR (supporting multiple languages and hand written documents), or are sent to a vision-capable LLM.

This tutorial downloads openly licensed files from Wikimedia Commons: Swap in your own meeting recordings, demos or slides to try it on your data.

In [25]:
import os
import urllib.parse
import urllib.request

COMMONS = "https://upload.wikimedia.org/wikipedia/commons"
media_urls = {
    # Spoken Wikipedia article, public domain
    "audio": f"{COMMONS}/2/27/Wikipedia_-_Superintelligence_-_Paths%2C_Dangers%2C_Strategies_%28spoken_by_AI_voice%29.mp3",
    # Diagram of a generative AI agent, CC BY-SA 4.0
    "image": f"{COMMONS}/1/16/GenAI_Agent.png",
    # Photo of a vegetable stall in Wolverhampton Market, CC BY-SA 2.0
    "image_market": f"{COMMONS}/f/f9/Lewis%27s_vegetables_in_Wolverhampton_Market_-_geograph.org.uk_-_1522285.jpg",
}

os.makedirs("samples", exist_ok=True)
media_files = {}
for kind, url in media_urls.items():
    path = os.path.join("samples", urllib.parse.unquote(os.path.basename(url)))
    if not os.path.exists(path):
        request = urllib.request.Request(url, headers={"User-Agent": "h2o-tutorial/1.0"})  # Wikimedia requires a User-Agent
        with urllib.request.urlopen(request) as response, open(path, "wb") as out:
            out.write(response.read())
    media_files[kind] = path

print("Files ready:", media_files)

Files ready: {'audio': 'samples/Wikipedia_-_Superintelligence_-_Paths,_Dangers,_Strategies_(spoken_by_AI_voice).mp3', 'image': 'samples/GenAI_Agent.png', 'image_market': "samples/Lewis's_vegetables_in_Wolverhampton_Market_-_geograph.org.uk_-_1522285.jpg"}


### Preview the files
Look at the diagram and play the recordings first, so you can judge the answers h2oGPTe gives later. The players stream from Wikimedia Commons, so nothing is embedded in the notebook.

In [ ]:
from IPython.display import Audio, Image, display

for kind, url in media_urls.items():
    if kind.startswith("image"):
        display(Image(url=url, width=500))
    elif kind == "audio":
        display(Audio(url=url))

In [ ]:
media_collection_id = h2ogpte.create_collection(
    name="Audio and images",
)

upload_ids = []
for kind, path in media_files.items():
    with open(path, "rb") as f:
        upload_ids.append(h2ogpte.upload(os.path.basename(path), f))

if upload_ids:
    h2ogpte.ingest_uploads(
        media_collection_id,
        upload_ids,
        audio_input_language=None,  # spoken-language hint such as "en" or "fr"; None lets the model detect it
        ocr_model="auto",           # text extraction from images and scanned pages
        handwriting_check=True,     # switch to a handwriting model when needed
        gen_doc_summaries=True,
    )

Models that can look at images are listed below. Turn on vision to let the LLM see the pictures and slides in your documents in addition to the extracted text.

In [27]:
h2ogpte.get_vision_capable_llm_names()

['amazon/nova-2-lite',
 'claude-haiku-4-5',
 'claude-opus-4-6',
 'claude-opus-4-7',
 'claude-opus-4-8',
 'claude-opus-5',
 'claude-opus-5-5',
 'claude-sonnet-4-6',
 'claude-sonnet-5',
 'claude-sonnet-5-5',
 'google/gemma-3-12b-it',
 'google/gemma-3-27b-it',
 'google/gemma-3-4b-it',
 'gpt-5.6-luna',
 'gpt-5.6-sol',
 'gpt-5.6-terra',
 'gpt-6-astra',
 'meta-llama/Llama-4-Maverick-17B-128E-Instruct',
 'meta-llama/Llama-4-Scout-17B-16E-Instruct',
 'mistralai/Magistral-Small-2509',
 'mistralai/Ministral-3-14B-Instruct',
 'mistralai/Ministral-3-3B-Instruct',
 'mistralai/Ministral-3-8B-Instruct',
 'mistralai/Mistral-Large-3-675B-Instruct',
 'moonshotai/Kimi-K2.5',
 'moonshotai/Kimi-K3',
 'nvidia/Nemotron-Nano-12B-v2-VL',
 'pixtral-large-2502',
 'Qwen/Qwen3-VL-235B-A22B-Instruct',
 'xai/grok-4.6']

In [28]:
media_chat_id = h2ogpte.create_chat_session(media_collection_id)

with h2ogpte.connect(media_chat_id) as session:
    media_answer = session.query(
        "Summarize each file in a few sentences. For the recordings, list the main topics in the order they are discussed. For each image, describe what it shows.",
        llm=llm,
        llm_args={"enable_vision": "auto"},  # "on" always sends images to a vision LLM
        rag_config={"rag_type": "rag+"},
        timeout=900,
    )

print(media_answer.content)

According to the documents, images, and audio transcript provided in this conversation, here is a summary of each file:

---

### 1. 📄 GenAI_Agent.pdf — Architecture Diagram
This image depicts a technical architecture diagram of a **GenAI Agent**. It illustrates a pipeline that begins with an external **Prompt**, flows through a **Preprocessing** stage, into a **Large Language Model (LLM)**, and then through a **Postprocessing** stage to produce an output. Three supporting resources — **Data**, **Tools**, and **Other Models** — are connected to the pipeline, indicating they can be leveraged at multiple stages. A feedback loop is also shown from the output back to the Preprocessing stage.

---

### 2. 🖼️ Lewis's Vegetables in Wolverhampton Market (Image)
This image shows an **outdoor vegetable market stall**, likely in Wolverhampton Market. Two older women with **wheeled shopping trolleys** are browsing the stall, which displays a variety of fresh produce including **carrots, cauliflowe

## Using agents in h2oGPTe

H2O Agents let the LLM plan and complete multi-step tasks, using tools such as code execution and web search, rather than only answering from retrieved text.

In [29]:
with h2ogpte.connect(chat_session_id) as session:
    agent_answer = session.query(
        "Write a short checklist for evaluating whether a RAG application is ready for production.",
        llm=llm,
        rag_config={"rag_type": "agent_only"},  # run the query with the agent
        timeout=600
    )

print(agent_answer.content)

<think> The user wants a short checklist for evaluating whether a RAG application is ready for production. I must base my answer ONLY on:
1. The chat history
2. Any images given (none in this case)
3. The document text provided in the context above

Let me review what information is available:

From the documents provided, I can see:
- The documents are navigation menus/table of contents for Enterprise h2oGPTe documentation
- They list various features related to evaluation, feedback, models, etc.
- Key evaluation-related features mentioned:
  - "Run an Evaluator on a Collection"
  - "Run an Evaluator on a Chat"
  - "Add a new evaluation dashboard"
  - "View and export feedback"
  - "Run self-tests"
  - "Supported LLMs"
  - "Prompt template settings"
  - "Create an Extractor for a document"

From the chat history, I can see previous discussions about:
- Features that help evaluate answer quality in Enterprise h2oGPTe
- The user has been asking about evaluation features

The user wants 

### Discover the available agents and tools
Enterprise h2oGPTe ships several built-in agents, including the more autonomous Super Agent, and you can add your own. List what your instance offers before choosing.

In [30]:
agents = h2ogpte.get_all_agents()

for agent in agents:
    print(f"{agent['value']:<30} {agent['label']:<30} type={agent['type']:<10} enabled={agent['enabled']}")

general                        General Agent                  type=built_in   enabled=True
deep_research                  Deep Research                  type=built_in   enabled=True
deep_research_zero             Deep Research Zero             type=built_in   enabled=True
data_science                   Data Science Agent             type=built_in   enabled=True
search                         Web search                     type=built_in   enabled=True
tool_builder                   Tool Builder                   type=built_in   enabled=True
agents_builder                 Agents Builder                 type=built_in   enabled=True
mcp_tool_runner                MCP-only Runner                type=built_in   enabled=True
coder                          Code Agent                     type=built_in   enabled=True


In [31]:
agent_tools = h2ogpte.get_agent_tools_dict()
print(len(agent_tools), "tools available")

for name, info in agent_tools.items():
    print("-", name)

31 tools available
- advanced_reasoning.py
- agents_builder.py
- agents_runner.py
- ask_question_about_documents.py
- audio_video_transcription.py
- browser_agent.py
- cancel_background_job.py
- convert_document_to_text.py
- download_web_video.py
- github.py
- global
- google_search.py
- driverless_ai_data_science.py
- image_generation.py
- internet
- intranet
- claude_tool_runner.py
- mermaid_renderer.py
- news_query.py
- python
- rag_text
- rag_vision
- scholar_papers_query.py
- screenshot_webpage.py
- shell
- claude_tool_creator.py
- unified_search.py
- wayback_machine_search.py
- query_to_web_image.py
- wikipedia_article.py
- wolfram_alpha_math_science_query.py


### Choose the agent, effort and tools
* `agent_accuracy` - `quick`, `basic`, `standard` or `maximum`. Higher means more turns and verification, but takes longer.
* `agent_tools` - `"auto"`, `"all"`, or a list of tool names from the previous cell.
* `agent_type` - leave on `"auto"` to let h2oGPTe route the task, or pass the `value` of one of the agents listed above.

In [32]:
with h2ogpte.connect(chat_session_id) as session:
    research = session.query(
        "Research how teams measure RAG quality in production. Compare at least three approaches "
        "in a table, then write a one-page recommendation with sources.",
        llm=llm,
        llm_args={
            "use_agent": True,
            "agent_type": "deep_research",
            "agent_accuracy": "maximum",
            "agent_tools": "auto",
            "agent_total_timeout": 1800,
        },
        timeout=1800,
    )

print(research.content)

Using agent: auto
# How Teams Measure RAG Quality in Production

Teams that run retrieval-augmented generation (RAG) in production generally do not rely on a single score. They combine several kinds of measurement, because each one covers a different failure mode:

- **Offline golden-set evaluation** checks retrieval and answer correctness against curated references.
- **Reference-free LLM-as-judge scoring** monitors live traffic without needing labelled answers.
- **User feedback and behaviour signals** show whether answers are actually useful.
- **Periodic human audits** calibrate everything else.

Platforms such as Enterprise h2oGPTe package parts of this into built-in evaluators, dashboards and feedback exports [docs.h2o.ai](https://docs.h2o.ai/enterprise-h2ogpte/guide/chats/evaluate-a-chat)[docs.h2o.ai](https://docs.h2o.ai/enterprise-h2ogpte/guide/collections/evaluate-a-collection)[docs.h2o.ai](https://docs.h2o.ai/enterprise-h2ogpte/guide/evaluation#add-a-new-evaluation-dashboard)

## Connect your own MCP server
[Model Context Protocol](https://modelcontextprotocol.io) (MCP) servers give agents new tools, such as querying an internal API, a ticketing system, or a code repository. Enterprise h2oGPTe supports:

* `remote_mcp` - a server you reach over a URL
* `local_mcp` - a server packaged as a `.zip` that runs inside the agent sandbox

This example uses DeepWiki's public remote MCP server, which needs no credentials.

### Inspect before you add
Inspection lists the server's tools and runs a security check.

In [35]:
mcp_servers = {"deepwiki": {"url": "https://mcp.deepwiki.com/mcp"}}

inspection = h2ogpte.inspect_mcp_tool(servers=mcp_servers, model="claude-sonnet-5-5")
print("Status:", inspection["status"])

for server in inspection["servers"]:
    print(f"{server['server_name']}: {server['tool_count']} tools")
    for tool in server["tools"]:
        print("  -", tool["name"], "|", tool["description"][:80])
    if server.get("security"):
        print("  Safe:", server["security"]["is_safe"], "| Warnings:", server["security"]["warnings"])

Status: success
deepwiki: 3 tools
  - ask_wiki_question | Ask any question about a GitHub repository's codebase and get an AI-powered answ
  - read_wiki_contents | View documentation about a GitHub repository.
  - read_wiki_structure | Get a list of documentation topics for a GitHub repository.
  Safe: True | Warnings: ['LLM finding: Remote server code is not auditable; trust placed entirely in third-party operator', 'LLM finding: Potential data exposure: user questions and repository identifiers are sent to an external service', 'LLM finding: Indirect prompt injection risk via wiki/repository content returned by tools', 'LLM finding: No authentication configured, so no client-side access control or per-user scoping']


### Add it as an agent tool

In [36]:
mcp_tool_ids = h2ogpte.add_custom_agent_tool(
    tool_type="remote_mcp",
    tool_args={
        "mcp_config_json": json.dumps({"mcpServers": mcp_servers}),
        "enable_by_default": True,
    },
)
print("MCP tool IDs:", mcp_tool_ids)

h2ogpte.list_mcp_tools(config_type="runner", server_names=["deepwiki"])

MCP tool IDs: ['1cef99d2-7b61-4c57-a62a-d083b551ddc5']


{'status': 'success', 'servers': {}, 'total_servers': 0, 'total_tools': 0}

### Use it in an agent task

In [38]:
with h2ogpte.connect(chat_session_id) as session:
    mcp_answer = session.query(
        "Use the DeepWiki tools to explain how the h2oai/h2o-3 repository is structured.",
        llm=llm,
        llm_args={"use_agent": True, "agent_tools": "all", "agent_accuracy": "standard"},
        rag_config={"rag_type": "agent_only"},
        timeout=900,
    )

print(mcp_answer.content)

# How the h2oai/h2o-3 Repository Is Structured

h2o-3 is a **multi-module Gradle project** made up of these layers:

- **A Java core** (`h2o-core`) provides distributed computing and data storage.
- **Algorithm modules** (`h2o-algos`) sit on top of the core.
- **A standalone scoring runtime** (`h2o-genmodel`) runs exported models without a cluster.
- **Plug-in modules** add storage back-ends (`h2o-persist-*`), file formats (`h2o-parsers`), Hadoop support (`h2o-hadoop-*`) and extras (`h2o-extensions`).
- **Client front-ends** (`h2o-py`, `h2o-r`, `h2o-web`) talk to the Java backend through a REST API. Their client code is generated from the backend's schemas by `h2o-bindings`.
- **Assembly modules** (`h2o-assemblies`) package all of this into runnable JARs.

All of this comes from the DeepWiki MCP server (`https://mcp.deepwiki.com/mcp`, DeepWiki v2.14.3), which was registered in the uploaded `remote_mcp_tools.json` [mcp.deepwiki.com](https://mcp.deepwiki.com/mcp). I used all three of its

To add a server that runs locally in the sandbox, zip it and upload it. Secrets such as API tokens are stored with `add_agent_key` and linked to the tool with `assign_agent_key_for_tool`, so they never appear in your code.

```python
h2ogpte.add_custom_agent_tool(
    tool_type="local_mcp",
    tool_args={"tool_name": "my_mcp", "description": "my mcp tool description and use", "enable_by_default": True},
    custom_tool_path="my_mcp_server.zip",
)
```

You can also package a full agent built with LangGraph, CrewAI, OpenAI or Claude with `h2ogpte.add_custom_agent`, and call it by passing its name as `agent_type`.

## Schedule recurring tasks
A scheduled task runs a prompt, optionally with an agent and a collection, on a timetable. Each run creates a chat session you can open later, including any files the agent produced.

This example runs a weekly research agent. It finds the articles getting the most attention on generative AI and autonomous agents, links to them, and builds a short PowerPoint report.

### Describe it in plain language !
h2oGPTe can turn a description into a task specification.

In [39]:
spec = h2ogpte.generate_scheduled_task_spec(
    prompt="Every Monday at 8am, research the latest generative AI and autonomous agent news, "
           "find the most discussed articles, and build a short PowerPoint report with links.",
    timezone="UTC",
    use_agent=True,
)
spec

{'task': {'passed': True,
  'dialog': 'text',
  'name': 'Weekly Generative AI News PowerPoint Report',
  'message': 'Research the latest generative AI and autonomous agent news from the past week. Find the most discussed and widely shared articles across tech news sources, blogs, and social platforms. Then build a short PowerPoint (.pptx) report summarizing the top stories, each with a brief description and a clickable link to the source. Keep it concise — a handful of slides highlighting the most notable developments.',
  'schedule_type': 'recurring',
  'schedule_expression': '0 8 * * 1',
  'task_type': 'agent_task',
  'timezone': 'UTC',
  'response': "I've scheduled a weekly task every Monday at 8am UTC (I defaulted to the UTC timezone since none was specified) to research the latest generative AI and autonomous agent news and build a short PowerPoint report with links. Reply yes to proceed or no to cancel."},
 'passed': True,
 'dialog': 'text',
 'response': "I've scheduled a weekly 

### Create the task
`schedule_type` is `once`, `recurring` (cron) or `interval` (for example `"6 hours"`). `task_type` is `chat_completion` or `agent_task`.

In [40]:
task_prompt = (
    "Research the latest developments in generative AI and autonomous agents from the past week. "
    "Search the web and identify the articles, papers and posts getting the most attention. "
    "Write a short, easy-to-digest report as a PowerPoint file (report.pptx) of at most 6 slides: "
    "one slide of key takeaways, one slide per major trend, and a final slide listing the links to "
    "the most-discussed articles with a one-line summary each. "
    "Also summarize the key points in your reply."
)

task = h2ogpte.create_scheduled_task(
    name="tutorial-genai-agents-weekly-report",
    description="Weekly research report on generative AI and autonomous agents",
    schedule_type="recurring",
    schedule_expression="0 8 * * 1",  # 08:00 every Monday
    task_type="agent_task",
    message=task_prompt,
    timezone="UTC",
    llm=llm,
    llm_args={"use_agent": True, "agent_accuracy": "basic", "agent_total_timeout": 1800},
    max_executions=4,
    notification_email="current_user",  # email the run result to yourself
)

print(task.id, task.status, "next run:", task.next_execution_at)

45a54a12-8976-4d3a-9c67-48d0d2c17162 active next run: 2026-10-19 08:00:00+00:00


### Test it now and get the report
Trigger a run immediately instead of waiting for Monday. The agent can take several minutes to research and build the slides.

In [41]:
import time

trigger = h2ogpte.trigger_scheduled_task(task.id)
print("Started execution", trigger.execution_id)

for _ in range(180):
    executions = h2ogpte.list_scheduled_task_executions(task.id).items
    if executions and executions[0].status not in ("pending", "running"):
        break
    time.sleep(10)

run = executions[0]
print(run.status, run.started_at)
print(run.result_summary or run.error_message)

# The report is attached to the chat session created by the run
for f in h2ogpte.get_agent_server_files(run.chat_session_id):
    print(f)

Started execution 79d23239-d244-4981-9a52-349fbaeff730
completed 2026-10-12 16:10:00.297907+00:00
# Generative AI & Autonomous Agents: Week of October 5–12, 2026

I created **<a href="/file?id=facd1e3f-373d-4cc7-9175-0b4877f9588a&name=report.pptx" target="_blank">report.pptx</a>**, a 6-slide PowerPoint report on the most-discussed generative AI and autonomous agent stories of the past week. It has a key-takeaways slide, four trend slides, and a final slide with 12 clickable links and a one-line summary for each. The script that built it reopened the saved file and confirmed it has exactly 6 
{'id': '0bf18160-92b1-4651-b686-9f66c3d4fcf4', 'object': 'file', 'bytes': 7067, 'created_at': 1791821401, 'filename': 'mcp_tools_runner.zip', 'purpose': 'assistants', 'sha256': '15ba60fecae6d69a2b3410c05047e5d629077429f7bcfb02a67e579ea3d531b3'}
{'id': '30dfb0a5-5271-4fdc-981d-4bc8c0fac54c', 'object': 'file', 'bytes': 36, 'created_at': 1791821672, 'filename': '.terminate', 'purpose': 'assistants_out

### Pause, resume and review
Pause a task without deleting its history, then list all your tasks.

In [42]:
h2ogpte.update_scheduled_task(task.id, status="paused")

for t in h2ogpte.list_scheduled_tasks(name_filter="tutorial-").items:
    print(t.name, "|", t.status, "|", t.schedule_expression, "| runs:", t.execution_count)

tutorial-genai-agents-weekly-report | paused | 0 8 * * 1 | runs: 0


## Clean up
Delete the scheduled task, MCP tool, chat sessions and collections - the documents can be re-ingested at any time.

In [43]:
h2ogpte.delete_scheduled_tasks([task.id])
h2ogpte.delete_custom_agent_tool(mcp_tool_ids)
h2ogpte.delete_chat_sessions([chat_session_id, media_chat_id])
h2ogpte.delete_collections([collection_id, media_collection_id])

Job(id='4f8de78e-564e-4339-b844-d91ab9bf914f', name='Deleting collections', passed=1.0, failed=0.0, progress=1.0, completed=True, canceled=False, date=datetime.datetime(2026, 10, 13, 8, 31, 52, tzinfo=TzInfo(0)), kind=<JobKind.DeleteCollectionsJob: 'DeleteCollectionsJob'>, statuses=[JobStatus(id='93539325f0d14ccf87f005ba8312db70', status='Deleting indexes...')], errors=[], last_update_date=datetime.datetime(2026, 10, 13, 8, 31, 57, tzinfo=TzInfo(0)), duration='5s', duration_seconds=5.0, start_time=1791880312.3622198, canceled_by='', cancel_reason='', timeout=3600.0, files_total=None, files_failed=None)